In [ ]:
import os
from pathlib import Path

_nb_dir = Path().resolve()
if _nb_dir.name == 'evaluation':
    os.chdir(_nb_dir.parent)
print(f'Working directory: {Path().resolve()}')

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import StratifiedKFold
from collections import defaultdict

plt.ioff()  # prevent inline backend from auto-displaying figures on creation

sys.path.insert(0, str(Path('.').resolve()))
from models.mamba_ribeiro import RibeiroMambaECGNet
from utils.data import get_record_paths_labels_patients

print('Imports OK')

## Configuration

In [ ]:
# -- Model checkpoints: Mamba d16 30-min, experiment 775456882433998769 --------
MLRUNS_BASE = 'mlruns/775456882433998769'
FOLD_MODEL_PATHS = {
    1: f'{MLRUNS_BASE}/6876c391bf26425db55a6a5a8cea0f1d/artifacts/model_mamba_d16_30min_fold1.pt',
    2: f'{MLRUNS_BASE}/0c6276dabc7144dbbb3097fdae00b254/artifacts/model_mamba_d16_30min_fold2.pt',
    3: f'{MLRUNS_BASE}/96b3d4d5c1da4657b24eaa27859d6ead/artifacts/model_mamba_d16_30min_fold3.pt',
}

N_MAMBA_BLOCKS = 4
D_STATE        = 16
WINDOW_MINUTES = 30
FS             = 500
DEVICE_NR      = 0
THRESHOLD      = 0.5
N_FOLDS        = 3
RANDOM_STATE   = 42

# -- Data paths - same as training (all 5 MIMIC subdirs 30-34 + Snowflake) ----
AF_DIR_SNOW = '/home/maria/data/AF_prediction_data/AF_prediction_project_lead_II_v2/snowflake/Pre_AF_SR/'
SR_DIR_SNOW = '/home/maria/data/AF_prediction_data/AF_prediction_project_lead_II_v2/snowflake/Pure_SR_Controls/'
MIMIC_BASE  = '/home/maria/data/AF_prediction_data/AF_prediction_project_lead_II_v2/mimic/'
MIMIC_SUBDIRS = [
    ('30/Pre_AF_SR/', '30/Pure_SR_Controls/'),
    ('31/Pre_AF_SR/', '31/Pure_SR_Controls/'),
    ('32/Pre_AF_SR/', '32/Pure_SR_Controls/'),
    ('33/Pre_AF_SR/', '33/Pure_SR_Controls/'),
    ('34/Pre_AF_SR/', '34/Pure_SR_Controls/'),
]
USE_SNOW  = True
USE_MIMIC = True

VIEW_SECS  = 30    # seconds visible in the scrollable ECG view
SMOOTH_SEC = 2.0   # Gaussian smoothing sigma in seconds

device = torch.device(f'cuda:{DEVICE_NR}' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
for fold, path in FOLD_MODEL_PATHS.items():
    print(f'  Fold {fold}: {"OK" if os.path.exists(path) else "NOT FOUND"} -- {path}')

## Load data + reproduce fold splits

Builds `pid_to_paths` and `pid_to_fold` so we know which model to use for each patient.

In [ ]:
all_paths, all_labels, all_pids = [], [], []

if USE_SNOW:
    p, l, ids = get_record_paths_labels_patients(AF_DIR_SNOW, SR_DIR_SNOW, dataset_prefix='SF_')
    all_paths.append(p); all_labels.append(l); all_pids.append(ids)

if USE_MIMIC:
    for af_sub, sr_sub in MIMIC_SUBDIRS:
        p, l, ids = get_record_paths_labels_patients(
            MIMIC_BASE + af_sub, MIMIC_BASE + sr_sub, dataset_prefix='M_'
        )
        all_paths.append(p); all_labels.append(l); all_pids.append(ids)

file_paths  = np.concatenate(all_paths)
labels      = np.concatenate(all_labels)
patient_ids = np.concatenate(all_pids)

# pid -> list of file paths (some patients have multiple recordings)
pid_to_paths  = defaultdict(list)
pid_to_label  = {}
for path, label, pid in zip(file_paths, labels, patient_ids):
    pid_to_paths[pid].append(path)
    pid_to_label[pid] = int(label)

# Reproduce fold assignment (same as training)
unique_patients   = np.array(list(pid_to_label.keys()))
unique_pat_labels = np.array([pid_to_label[p] for p in unique_patients])

skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
patient_fold = np.full(len(unique_patients), -1, dtype=int)

dataset_prefixes = []
if USE_SNOW:  dataset_prefixes.append('SF_')
if USE_MIMIC: dataset_prefixes.append('M_')

for prefix in dataset_prefixes:
    mask        = np.array([str(p).startswith(prefix) for p in unique_patients])
    ds_patients = unique_patients[mask]
    ds_labels   = unique_pat_labels[mask]
    for fold_idx, (_, test_idx) in enumerate(skf.split(ds_patients, ds_labels)):
        patient_fold[np.where(mask)[0][test_idx]] = fold_idx

pid_to_fold = {
    pid: (int(patient_fold[i]) + 1) if patient_fold[i] >= 0 else 1
    for i, pid in enumerate(unique_patients)
}

print(f'Total patients: {len(pid_to_fold)}')
print(f'  AF: {sum(1 for v in pid_to_label.values() if v==1)}')
print(f'  SR: {sum(1 for v in pid_to_label.values() if v==0)}')

## Saliency helpers

In [ ]:
def load_model(path, device):
    model = RibeiroMambaECGNet(
        n_classes=1, duration_minutes=float(WINDOW_MINUTES), fs=FS,
        kernel_size=16, n_mamba_blocks=N_MAMBA_BLOCKS,
        pre_mamba_stride=4, conv_dropout=0.3,
        mamba_dropout=0.2, final_dropout=0.5,
        mamba_d_state=D_STATE,
    )
    model.load_state_dict(torch.load(path, map_location=device, weights_only=False))
    model.to(device).eval()
    return model


def get_fs_from_hea(path):
    try:
        with open(path + '.hea') as f:
            parts = f.readline().split()
            if len(parts) >= 3:
                return float(parts[2])
    except Exception:
        pass
    return 500.0


@torch.no_grad()
def get_prob(model, chunk, device):
    x = torch.tensor(chunk, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    return torch.sigmoid(model(x)).item()


# -- Grad-CAM -----------------------------------------------------------------
def compute_gradcam(model, chunk, device):
    activations, gradients = {}, {}

    def fwd_hook(m, inp, out):
        activations['feat'] = out

    def bwd_hook(m, grad_in, grad_out):
        gradients['feat'] = grad_out[0]

    h1 = model.stage3.conv2.register_forward_hook(fwd_hook)
    h2 = model.stage3.conv2.register_full_backward_hook(bwd_hook)

    try:
        x = torch.tensor(chunk, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        model.zero_grad()
        logit = model(x)
        prob  = torch.sigmoid(logit).item()
        logit.backward()

        act  = activations['feat'].squeeze(0).detach()
        grad = gradients['feat'].squeeze(0).detach()
        weights = grad.mean(dim=-1)
        cam     = torch.relu((weights[:, None] * act).sum(0))
        cam     = cam.cpu().float().numpy()

        L = len(chunk)
        saliency = np.interp(
            np.arange(L),
            np.linspace(0, L - 1, len(cam)),
            cam
        ).astype(np.float32)
        sal_max = saliency.max()
        if sal_max > 1e-8:
            saliency /= sal_max
    finally:
        h1.remove()
        h2.remove()

    return saliency, prob


# -- Integrated Gradients -----------------------------------------------------
def compute_integrated_gradients(model, chunk, device,
                                  n_steps=25, baseline='zero', status_cb=None):
    L    = len(chunk)
    base = (np.zeros(L, dtype=np.float32) if baseline == 'zero'
            else np.full(L, chunk.mean(), dtype=np.float32))

    alphas = np.linspace(0, 1, n_steps + 1)
    grads  = []
    prob   = 0.0

    for step, alpha in enumerate(alphas):
        if status_cb is not None and step % 5 == 0:
            status_cb(f'Integrated Gradients -- step {step}/{n_steps}...')
        interp = (base + alpha * (chunk - base)).astype(np.float32)
        x = torch.tensor(interp, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        x.requires_grad_(True)
        model.zero_grad()
        logit = model(x)
        if step == n_steps:
            prob = torch.sigmoid(logit).item()
        logit.backward()
        grads.append(x.grad.detach().squeeze().cpu().float().numpy())

    avg_grads = np.mean(grads, axis=0)
    ig = np.abs((chunk - base) * avg_grads)
    ig_max = ig.max()
    if ig_max > 1e-8:
        ig /= ig_max
    return ig, prob


# -- Mamba Delta proxy --------------------------------------------------------
def compute_delta_profile(model, chunk, device):
    L        = len(chunk)
    captured = {}
    hooks    = []

    for idx in range(len(model.mamba_blocks)):
        def make_hook(i):
            def hook(m, inp, out):
                diff = (out - inp[0]).detach().cpu()
                captured[i] = diff.norm(dim=1).squeeze(0).float().numpy()
            return hook
        h = model.mamba_blocks[idx].register_forward_hook(make_hook(idx))
        hooks.append(h)

    with torch.no_grad():
        x    = torch.tensor(chunk, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        prob = torch.sigmoid(model(x)).item()

    for h in hooks:
        h.remove()

    block_profiles = []
    for idx in range(len(model.mamba_blocks)):
        if idx not in captured:
            continue
        profile = captured[idx]
        block_profiles.append(
            np.interp(
                np.arange(L),
                np.linspace(0, L - 1, len(profile)),
                profile
            ).astype(np.float32)
        )

    if not block_profiles:
        return np.zeros(L, dtype=np.float32), prob

    delta = np.mean(block_profiles, axis=0)
    d_max = delta.max()
    if d_max > 1e-8:
        delta /= d_max
    return delta.astype(np.float32), prob


# -- Grad-CAM x Delta ---------------------------------------------------------
def compute_gradcam_x_delta(model, chunk, device):
    gradcam, prob = compute_gradcam(model, chunk, device)
    delta,   _    = compute_delta_profile(model, chunk, device)
    combined = gradcam * delta
    c_max    = combined.max()
    if c_max > 1e-8:
        combined /= c_max
    return combined.astype(np.float32), prob


def smooth(saliency, sigma_s, fs):
    return gaussian_filter1d(saliency, sigma=max(1, int(sigma_s * fs)))


print('Saliency helpers ready.')

## Visualization helpers

In [ ]:
# -- Saliency colormap: dark green -> orange -> dark red ----------------------
_SALIENCY_CMAP = LinearSegmentedColormap.from_list(
    'sal', [(0.0, '#1a5c34'), (0.15, '#f39c12'), (1.0, '#c0392b')]
)

# -- ECG paper grid -----------------------------------------------------------
_MINOR_T = 0.04
_MAJOR_T = 0.20
_BG      = '#fff8f8'
_C_MIN   = '#ffcccc'
_C_MAJ   = '#ff8888'


def _nice_amp_steps(y_min, y_max, n_minor=10, n_major=4):
    span = max(y_max - y_min, 1e-6)
    def nice(raw):
        mag = 10 ** np.floor(np.log10(raw))
        for f in [1, 2, 5, 10]:
            if f * mag >= raw:
                return f * mag
        return mag * 10
    return nice(span / n_minor), nice(span / n_major)


def _add_ecg_grid(ax, x_start, x_end, y_min, y_max):
    ax.set_facecolor(_BG)
    for xi in np.arange(np.floor(x_start / _MINOR_T) * _MINOR_T, x_end + _MINOR_T, _MINOR_T):
        ax.axvline(xi, color=_C_MIN, linewidth=0.3, zorder=0)
    for xi in np.arange(np.floor(x_start / _MAJOR_T) * _MAJOR_T, x_end + _MAJOR_T, _MAJOR_T):
        ax.axvline(xi, color=_C_MAJ, linewidth=0.8, zorder=0)
    minor_a, major_a = _nice_amp_steps(y_min, y_max)
    for yi in np.arange(np.floor(y_min / minor_a) * minor_a, y_max + minor_a, minor_a):
        ax.axhline(yi, color=_C_MIN, linewidth=0.3, zorder=0)
    for yi in np.arange(np.floor(y_min / major_a) * major_a, y_max + major_a, major_a):
        ax.axhline(yi, color=_C_MAJ, linewidth=0.8, zorder=0)


# -- Single-method plot -------------------------------------------------------
def plot_ecg_saliency(signal, saliency, fs, view_start_s,
                      view_len_s=10, prob=None, title='', method_label='Saliency',
                      show_grid=True):
    L       = len(signal)
    total_s = L / fs
    i0      = int(view_start_s * fs)
    i1      = min(i0 + int(view_len_s * fs), L)
    t_view  = np.arange(i0, i1) / fs
    sig_v   = signal[i0:i1]
    sal_v   = saliency[i0:i1]
    ypad    = max(0.3, (sig_v.max() - sig_v.min()) * 0.1)
    y_min   = sig_v.min() - ypad
    y_max   = sig_v.max() + ypad

    fig = plt.figure(figsize=(16, 6))
    gs  = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.5)
    ax_ecg = fig.add_subplot(gs[0])

    if show_grid and len(t_view) > 0:
        _add_ecg_grid(ax_ecg, t_view[0], t_view[-1], y_min, y_max)

    if len(t_view) > 1:
        pts  = np.array([t_view, sig_v]).T.reshape(-1, 1, 2)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
        lc   = LineCollection(segs, cmap=_SALIENCY_CMAP,
                              norm=plt.Normalize(0, 1), linewidth=1.0, zorder=2)
        lc.set_array(sal_v[:-1])
        ax_ecg.add_collection(lc)
        cbar = plt.colorbar(lc, ax=ax_ecg, pad=0.01, fraction=0.015)
        cbar.set_label('Saliency', fontsize=9)

    ax_ecg.set_xlim(t_view[0], t_view[-1])
    ax_ecg.set_ylim(y_min, y_max)
    ax_ecg.set_xlabel('Time (s)', fontsize=9)
    ax_ecg.set_ylabel('Amplitude', fontsize=9)

    prob_str  = (f'  |  AF prob = {prob:.3f}  ({"AF" if prob and prob >= THRESHOLD else "SR"})'
                 if prob is not None else '')
    grid_note = '  |  grid: 0.04 s / 0.2 s (25 mm/s)' if show_grid else ''
    ax_ecg.set_title(
        f'{title}{prob_str}\n'
        f'Viewing {view_start_s:.0f}s - {view_start_s + view_len_s:.0f}s  '
        f'(of {total_s:.0f}s total){grid_note}   |   colour: dark red = high {method_label} saliency',
        fontsize=10
    )

    ax_sal = fig.add_subplot(gs[1])
    t_full = np.arange(L) / fs
    ax_sal.fill_between(t_full, saliency, color='tomato', alpha=0.65)
    ax_sal.add_patch(Rectangle(
        (view_start_s, 0), view_len_s, 1.05,
        alpha=0.25, color='royalblue', zorder=3, clip_on=False
    ))
    ax_sal.axvline(view_start_s,              color='royalblue', lw=1.5, zorder=4)
    ax_sal.axvline(view_start_s + view_len_s, color='royalblue', lw=1.5, zorder=4)
    ax_sal.set_xlim(0, total_s)
    ax_sal.set_ylim(0, 1.05)
    ax_sal.set_xlabel('Time in window (s)', fontsize=9)
    ax_sal.set_ylabel('Saliency', fontsize=9)
    ax_sal.set_title(f'{method_label} overview - blue = current view', fontsize=9)

    plt.tight_layout()
    return fig


# -- Multi-method comparison plot ---------------------------------------------
def plot_method_comparison(signal, results_dict, fs, view_start_s,
                            view_len_s=10, show_grid=True):
    n  = len(results_dict)
    L  = len(signal)
    i0 = int(view_start_s * fs)
    i1 = min(i0 + int(view_len_s * fs), L)
    t_view = np.arange(i0, i1) / fs
    sig_v  = signal[i0:i1]
    ypad   = max(0.3, (sig_v.max() - sig_v.min()) * 0.12)
    y_min  = sig_v.min() - ypad
    y_max  = sig_v.max() + ypad

    fig, axes = plt.subplots(n, 1, figsize=(16, 3.6 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, (name, (saliency, prob)) in zip(axes, results_dict.items()):
        if show_grid and len(t_view) > 0:
            _add_ecg_grid(ax, t_view[0], t_view[-1], y_min, y_max)

        sal_v = saliency[i0:i1]
        if len(t_view) > 1:
            pts  = np.array([t_view, sig_v]).T.reshape(-1, 1, 2)
            segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
            lc   = LineCollection(segs, cmap=_SALIENCY_CMAP,
                                  norm=plt.Normalize(0, 1), linewidth=1.0, zorder=2)
            lc.set_array(sal_v[:-1])
            ax.add_collection(lc)
            plt.colorbar(lc, ax=ax, pad=0.01, fraction=0.012, label='Saliency')

        ax.set_xlim(t_view[0], t_view[-1])
        ax.set_ylim(y_min, y_max)
        ax.set_ylabel('Amplitude', fontsize=8)
        outcome = 'AF' if prob >= THRESHOLD else 'SR'
        ax.set_title(f'{name}  |  prob = {prob:.3f} ({outcome})', fontsize=10, pad=3)

    axes[-1].set_xlabel('Time (s)', fontsize=9)
    plt.suptitle(
        f'Saliency method comparison  --  {view_start_s:.0f}s to '
        f'{view_start_s + view_len_s:.0f}s   (dark red = high influence, green = low)',
        fontsize=11, y=1.10
    )
    plt.tight_layout()
    return fig


print('Visualization helpers ready.')
print('  Colormap: dark green (low, 0-15%) -> orange -> dark red (high)')
print('  ECG grid: time 0.04 s / 0.2 s (25 mm/s) | amplitude auto-scaled to signal range')

## Interactive Saliency Viewer

**Methods available:**
- **Grad-CAM** — fast (~1s), hooks `model.stage3.conv2`, gradients propagate through Mamba
- **Integrated Gradients** — medium (~30s), axiomatically sound, full end-to-end attribution
- **Mamba Delta** — fast (~1s), `||output - input||_2` per MambaResidualBlock position (proxy for Delta)
- **Grad-CAM x Delta** — fast (~2s), element-wise product: discriminative AND actively updated regions
- **All methods** — runs all 4 and shows a side-by-side comparison

1. Pick a patient and which 30-min window to analyse
2. Click **Analyse**
3. Scroll through the 30-min window with the slider or play button

**Raw amplitude:** tick the checkbox to display original (non-normalized) signal amplitude. The model always receives the z-score normalized signal — this only affects the y-axis display.

In [ ]:
def _build_widgets():
    af_pids  = sorted(p for p, l in pid_to_label.items() if l == 1)
    sr_pids  = sorted(p for p, l in pid_to_label.items() if l == 0)
    all_opts = [(f'{p}  [AF]', p) for p in af_pids] + \
               [(f'{p}  [SR]', p) for p in sr_pids]
    return dict(
        patient  = widgets.Dropdown(options=all_opts, description='Patient:',
                                    layout=widgets.Layout(width='380px')),
        window   = widgets.Dropdown(
            options=[('Best (highest AF prob)', 'best'),
                     ('Window 0  --   0-30 min', 0),
                     ('Window 1  --  30-60 min', 1),
                     ('Window 2  --  60-90 min', 2),
                     ('Window 3  -- 90-120 min', 3)],
            description='Window:', layout=widgets.Layout(width='280px')),
        method   = widgets.Dropdown(
            options=[
                ('Grad-CAM              (fast ~1s   |  ResNet features, Mamba via backprop)',   'gradcam'),
                ('Integrated Gradients  (medium ~30s|  full model, axiomatically grounded)',    'ig'),
                ('Mamba Delta           (fast ~1s   |  Delta proxy: ||output-input||_2 per block)', 'delta'),
                ('Grad-CAM x Delta      (fast ~2s   |  combined: discriminative + Mamba Delta)',   'gradcam_x_delta'),
                ('All methods -- side-by-side comparison',                                      'all'),
            ],
            description='Method:', layout=widgets.Layout(width='620px'),
            style={'description_width': '70px'}),
        view_len = widgets.Dropdown(
            options=[('5 s  -- very detailed (one beat cluster)', 5),
                     ('10 s -- standard ECG strip (default)',    10),
                     ('20 s -- wider context',                   20),
                     ('30 s -- full model window',               30)],
            value=10, description='View length:',
            layout=widgets.Layout(width='340px'),
            style={'description_width': '100px'}),
        smooth   = widgets.Checkbox(value=True, description=f'Smooth (sigma={SMOOTH_SEC}s)',
                                    layout=widgets.Layout(width='185px')),
        grid     = widgets.Checkbox(value=True, description='ECG grid (25 mm/s)',
                                    layout=widgets.Layout(width='185px')),
        raw      = widgets.Checkbox(value=False, description='Raw amplitude',
                                    layout=widgets.Layout(width='160px')),
        ig_steps = widgets.IntSlider(min=10, max=50, step=5, value=25,
                                     description='IG steps:',
                                     layout=widgets.Layout(width='340px'),
                                     style={'description_width': '80px'}),
        btn    = widgets.Button(description='Analyse', button_style='primary',
                                layout=widgets.Layout(width='120px', height='34px')),
        status = widgets.HTML(value=''),
        slider = widgets.IntSlider(min=0, max=1, step=1, value=0,
                                   continuous_update=False, description='Start (s):',
                                   layout=widgets.Layout(width='560px')),
        play   = widgets.Play(min=0, max=1, step=1, interval=700),
        img    = widgets.Image(format='png', width='100%'),
    )


# Always rebuild to pick up new widgets
globals()['_D16_30M_W'] = _build_widgets()
widgets.jslink((globals()['_D16_30M_W']['play'],   'value'),
               (globals()['_D16_30M_W']['slider'], 'value'))

_W         = globals()['_D16_30M_W']
w_patient  = _W['patient'];  w_window   = _W['window'];   w_method   = _W['method']
w_view_len = _W['view_len']; w_smooth   = _W['smooth'];   w_grid     = _W['grid']
w_raw      = _W['raw'];      w_ig_steps = _W['ig_steps']
w_btn      = _W['btn'];      w_status   = _W['status'];   w_slider   = _W['slider']
w_play     = _W['play'];     w_img      = _W['img']

state = dict(signal=None, raw_signal=None, saliency=None, prob=None, fs=FS,
             title='', method_label='Saliency', all_results={})


def set_status(msg, color='darkorange'):
    w_status.value = f'<span style="color:{color}">{msg}</span>'


def apply_smooth(sal, fs):
    sal = smooth(sal, SMOOTH_SEC, fs)
    m = sal.max()
    if m > 1e-8:
        sal /= m
    return sal


import io as _io


def draw(start_s):
    if state['signal'] is None:
        return
    vlen     = w_view_len.value
    show     = w_grid.value
    # Use raw amplitude signal for display if checkbox ticked
    disp_sig = state['raw_signal'] if (w_raw.value and state['raw_signal'] is not None) else state['signal']
    plt.close('all')
    if state['all_results']:
        # For all-methods view, substitute display signal into each result tuple
        disp_results = {k: (sal, prob) for k, (sal, prob) in state['all_results'].items()}
        fig = plot_method_comparison(
            disp_sig, disp_results, state['fs'],
            start_s, view_len_s=vlen, show_grid=show)
    else:
        fig = plot_ecg_saliency(
            disp_sig, state['saliency'], state['fs'],
            view_start_s=start_s, view_len_s=vlen,
            prob=state['prob'], title=state['title'],
            method_label=state['method_label'], show_grid=show)
    buf = _io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    w_img.value = buf.read()


def on_slider_change(change):
    draw(change['new'])


def on_view_len_change(change):
    if state['signal'] is None:
        return
    total_s   = len(state['signal']) / state['fs']
    max_start = max(0, int(total_s) - change['new'] - 1)
    w_slider.unobserve(on_slider_change, names='value')
    w_slider.max = max_start;  w_play.max = max_start
    w_slider.value = min(w_slider.value, max_start)
    w_slider.observe(on_slider_change, names='value')
    draw(w_slider.value)


def on_grid_change(change):
    draw(w_slider.value)


def on_raw_change(change):
    draw(w_slider.value)


# -- Method dispatch ----------------------------------------------------------
METHOD_LABELS = {
    'gradcam'        : 'Grad-CAM',
    'ig'             : 'Integrated Gradients',
    'delta'          : 'Mamba Delta',
    'gradcam_x_delta': 'Grad-CAM x Delta',
}


def run_method(key, model, chunk, device):
    if key == 'gradcam':
        return compute_gradcam(model, chunk, device)
    elif key == 'ig':
        return compute_integrated_gradients(
            model, chunk, device,
            n_steps=w_ig_steps.value,
            status_cb=set_status)
    elif key == 'delta':
        return compute_delta_profile(model, chunk, device)
    elif key == 'gradcam_x_delta':
        return compute_gradcam_x_delta(model, chunk, device)
    else:
        raise ValueError(f'Unknown method: {key}')


# -- Main callback ------------------------------------------------------------
def on_analyse(btn):
    pid   = w_patient.value
    paths = list(pid_to_paths.get(pid, []))
    if not paths:
        set_status(f'No recordings found for {pid}', 'red'); return

    fold = pid_to_fold.get(pid, 1)
    set_status(f'Loading fold-{fold} model (Mamba d16 30-min)...')
    model = load_model(FOLD_MODEL_PATHS[fold], device)

    path     = paths[0]
    fs       = int(get_fs_from_hea(path))
    raw      = np.fromfile(path + '.dat', dtype=np.float32)
    win_size = int(WINDOW_MINUTES * 60 * fs)
    starts   = list(range(0, len(raw) - win_size + 1, win_size))
    if not starts:
        set_status('Signal too short for one full window', 'red')
        del model; return

    win_choice = w_window.value
    if win_choice == 'best':
        set_status('Finding best window (highest AF probability)...')
        probs = []
        for s in starts:
            c = raw[s:s + win_size].copy()
            c = (c - c.mean()) / (c.std() + 1e-8)
            probs.append(get_prob(model, c, device))
        win_idx = int(np.argmax(probs))
    else:
        win_idx = min(int(win_choice), len(starts) - 1)

    # Raw (non-normalized) chunk -- for display only
    raw_chunk  = raw[starts[win_idx]:starts[win_idx] + win_size].copy()
    # Normalized chunk -- for model input
    chunk      = raw_chunk.copy()
    chunk      = (chunk - chunk.mean()) / (chunk.std() + 1e-8)

    s0m   = starts[win_idx] / (fs * 60)
    s1m   = (starts[win_idx] + win_size) / (fs * 60)
    lbl   = 'AF' if pid_to_label.get(pid) == 1 else 'SR'
    base_title = (f'{pid} | fold {fold} | window {win_idx} '
                  f'({s0m:.0f}-{s1m:.0f} min) | {lbl}')

    method    = w_method.value
    do_smooth = w_smooth.value

    if method == 'all':
        all_results = {}
        for key, label in METHOD_LABELS.items():
            set_status(f'Running {label}...')
            sal, prob = run_method(key, model, chunk, device)
            if do_smooth:
                sal = apply_smooth(sal, fs)
            all_results[label] = (sal, prob)
        state.update(signal=chunk, raw_signal=raw_chunk, saliency=None, prob=prob,
                     fs=fs, title=base_title, method_label='', all_results=all_results)
    else:
        label = METHOD_LABELS[method]
        set_status(f'Running {label}...')
        sal, prob = run_method(method, model, chunk, device)
        if do_smooth:
            sal = apply_smooth(sal, fs)
        state.update(signal=chunk, raw_signal=raw_chunk, saliency=sal, prob=prob,
                     title=f'{label} -- {base_title}',
                     method_label=label, fs=fs, all_results={})

    del model
    torch.cuda.empty_cache()

    total_s   = len(chunk) / fs
    vlen      = w_view_len.value
    max_start = max(0, int(total_s) - vlen - 1)
    outcome   = 'AF' if prob >= THRESHOLD else 'SR'
    set_status(
        f'Done!  AF prob = {prob:.3f} ({outcome})  |  '
        f'Scroll with the slider or use the play button.',
        color='green'
    )

    w_slider.unobserve(on_slider_change, names='value')
    w_slider.max = max_start;  w_play.max = max_start
    w_slider.step = max(1, vlen // 4)
    w_play.step   = w_slider.step
    w_slider.value = 0;        w_play.value = 0
    w_slider.observe(on_slider_change, names='value')
    draw(0)


# -- Register callbacks -------------------------------------------------------
w_slider.unobserve_all('value')
w_slider.observe(on_slider_change, names='value')

w_view_len.unobserve_all('value')
w_view_len.observe(on_view_len_change, names='value')

w_grid.unobserve_all('value')
w_grid.observe(on_grid_change, names='value')

w_raw.unobserve_all('value')
w_raw.observe(on_raw_change, names='value')

w_btn._click_handlers.callbacks.clear()
w_btn.on_click(on_analyse)

# -- Layout -------------------------------------------------------------------
display(widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px'>ECG Saliency Viewer -- Mamba d16 30-min (v2)</h3>"),
    widgets.HTML(
        "<small>"
        "Model: Mamba d_state=16, 30-min windows, trained on MIMIC subdirs 30-34 + Snowflake.<br>"
        "ECG line colour: <b style='color:#c0392b'>dark red</b> = high model attention &nbsp;|&nbsp; "
        "<b style='color:#27ae60'>green</b> = low attention."
        "&nbsp; Grid: standard ECG paper (25 mm/s, minor 0.04 s, major 0.2 s)."
        "&nbsp; <b>Raw amplitude</b>: displays original signal scale (model still uses normalized input)."
        "</small>"
    ),
    widgets.HTML("<hr style='margin:5px 0'>"),
    widgets.HBox([w_patient, w_window]),
    w_method,
    w_view_len,
    widgets.HBox([w_smooth, w_grid, w_raw, w_btn]),
    widgets.HTML(
        "<small style='color:gray'>IG steps applies to Integrated Gradients only "
        "(more steps = more accurate but slower):</small>"
    ),
    w_ig_steps,
    w_status,
    widgets.HTML("<hr style='margin:5px 0'>"),
    widgets.HBox([w_play, w_slider]),
    w_img,
]))